In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tools.utils import *
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import spacy

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# uncomment to download the model
# spacy.cli.download("en_core_web_sm")


In [ ]:
# prepare the spacy model and pipeline
nlp = spacy.load("en_core_web_sm")
nlp.disable_pipes("tagger", "parser", "attribute_ruler", "lemmatizer")
nlp.add_pipe('sentencizer')


In [ ]:
# next line is to check if MPS is available
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
# comment next line if you are want to run code on a GPU
#device = 'cuda' if torch.cuda.is_available() else device

# choose the embedding model: "nomic-embed-text-v1.5" | "qwen3-embedding-0.6b" | "embeddinggemma"
embedding_model_name = "bge-large-en-v1.5"

# load the embedding model and its task-instruction text prefix, and send it to the device
model, prefix = load_embedding_model(embedding_model_name, device=device)
print(device, embedding_model_name)

## Process TOUs dataset

In [ ]:
# set the path to the data folder
data_path = 'data/tous_data'
# process all the text files in the data folder
embeddings, df = process_data(data_path, nlp, model, prefix=prefix)

In [ ]:
# Save embeddings and metadata to disk
out_path = Path('processed_data/tous')
out_path.mkdir(exist_ok=True)
df.to_csv(out_path / 'metadata.tsv', index=False, sep='\t')
np.savetxt(out_path /'embedding_bge.tsv',embeddings,delimiter='\t')


## Process multi-genre dataset

In [ ]:
from pathlib import Path
txt_paths = list(Path('data/multigenre_data').rglob('**/*.txt'))
len(txt_paths)

In [ ]:
import re
sentence_modified = "This is a sentence mentioning Windows and Linux.".lower()
platform = "Windows"


In [ ]:
txt_paths[0].parent.parent.stem

In [ ]:
out_path = Path('processed_data/multigenre')
out_path.mkdir(exist_ok=True)

(out_path / f"metadata_{txt_paths[0].parent.parent.stem}_{txt_paths[0].stem.split('_')[0]}.tsv").is_file()

In [ ]:
out_path = Path('processed_data/multigenre/split')
out_path.mkdir(exist_ok=True)

for f in tqdm(txt_paths):    
    metadata, embeddings = [], []
    platform = f.stem.split('_')[0]
    tag = f.parent.parent.stem
    genre = f.parent.stem
    
    if (out_path / f"metadata_{tag}_{platform}.tsv").is_file(): continue

    with open(f) as in_txt:
        text = in_txt.read().strip()
        
    doc = nlp(text)
        
    for sentence  in doc.sents:
        if len(str(sentence)) < 10: continue

        sentence_modified = replace_named_entities(sentence) # remove named entities
        sentence_modified = sentence_modified.lower()#.replace(platform.lower(),'[mask]') # make double sure the platform name is masked    
        sentence_modified = re.sub(rf"\b{platform.lower()}\b", '[mask]', sentence_modified) # make double sure the platform name is masked using regex
        sentence_modified = remove_urls(sentence_modified) # remove urls
        metadata.append([f.stem,platform,genre,tag,sentence,sentence_modified])
        embeddings.append(model.encode(prefix + sentence_modified))

    df = pd.DataFrame(metadata, columns=['file_name','platform','genre','tag','sentence_original','sentence_modified'])    
    print(f'Embedded {len(df)} sentences...')
    print(f'Saving embeddings to', out_path /f'embedding_{tag}_{platform}.tsv')
    # Save embeddings and metadata to disk

    df.to_csv(out_path / f'metadata_{tag}_{platform}.tsv', index=False, sep='\t')
    np.savetxt(out_path /f'embedding_{tag}_{platform}.tsv',embeddings,delimiter='\t')


In [ ]:
out_path = Path('processed_data/multigenre')
out_path.mkdir(exist_ok=True)

In [ ]:
metadata_files = list((out_path / 'split').glob('metadata*.tsv'))

In [ ]:
metadata, embeddings = [], []
for m in metadata_files:
    metadata.append(pd.read_csv(m, sep='\t'))
    _, tag, platform = m.stem.split('_')
    embeddings.append(np.loadtxt((out_path / 'split') / f'embedding_{tag}_{platform}.tsv'))

    

In [ ]:
#np.concatenate(embeddings,axis=0).shape

In [ ]:
embeddings = np.concatenate([e for e in embeddings if e.any()],axis=0)
embeddings.shape

In [ ]:
metadata = pd.concat(metadata, axis=0, ignore_index=True)
metadata.shape

In [ ]:
metadata.head()

In [ ]:
metadata.to_csv(out_path / 'metadata.tsv', index=False, sep='\t')
np.savetxt(out_path / 'embedding.tsv',embeddings,delimiter='\t')


# Fin.